# Telco Churn EDA

Working through the IBM Telco dataset to understand what actually drives churn before building the model. Not going to over-clean this notebook — keeping the exploration style.

In [18]:
import sys
sys.path.insert(0, '../')
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data.load import load_raw

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 30)

df = load_raw()
print(df.shape)
df.head()

ModuleNotFoundError: No module named 'numpy'

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

TotalCharges has a few nulls — these are customers with tenure=0 (new customers who haven't been billed yet). I'll fill them with 0 in the loading step, which is what the data implies.

In [ ]:
print(f"Churn rate: {df['Churn'].mean():.2%}")
df['Churn'].value_counts()

~26.5% churn — moderately imbalanced. Not severe enough to throw off standard metrics entirely, but enough that precision/recall tradeoffs matter.

In [ ]:
# Churn by contract type — this is likely the single strongest signal
ct = df.groupby('Contract')['Churn'].agg(['mean', 'count'])
ct.columns = ['churn_rate', 'count']
ct.sort_values('churn_rate', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, ['Contract', 'PaymentMethod', 'InternetService']):
    rates = df.groupby(col)['Churn'].mean().sort_values(ascending=False)
    rates.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'Churn rate by {col}', fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('Churn rate')
    ax.tick_params(axis='x', rotation=30)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
plt.show()

Contract type is massive — month-to-month customers churn at ~43% vs 11% for one-year and just 3% for two-year. Payment method also matters: electronic check has notably higher churn. Fiber optic has higher churn than DSL, which is counterintuitive — probably a price sensitivity issue.

In [ ]:
# Tenure distribution by churn
fig, ax = plt.subplots(figsize=(9, 4))
df[df['Churn']==0]['tenure'].hist(bins=30, alpha=0.6, label='No churn', ax=ax, color='steelblue')
df[df['Churn']==1]['tenure'].hist(bins=30, alpha=0.6, label='Churn', ax=ax, color='salmon')
ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Count')
ax.set_title('Tenure distribution by churn status')
ax.legend()
plt.tight_layout()
plt.show()

Strong left-skew for churners — most people who leave do so in the first 12 months. Long-tenured customers almost never churn. This suggests tenure is a proxy for satisfaction/stickiness.

In [ ]:
# Monthly charges vs churn
fig, ax = plt.subplots(figsize=(8, 4))
df.boxplot(column='MonthlyCharges', by='Churn', ax=ax)
ax.set_title('Monthly Charges by Churn')
ax.set_xlabel('Churn (0=No, 1=Yes)')
ax.set_ylabel('Monthly Charges ($)')
plt.suptitle('')
plt.tight_layout()
plt.show()

print(df.groupby('Churn')['MonthlyCharges'].describe())

Churners pay more per month on average (~$74 vs ~$61). Combined with fiber optic correlation, there might be a price-to-value perception issue — customers with high charges who don't see value leave.

In [ ]:
# Service add-ons churn analysis
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                'TechSupport', 'StreamingTV', 'StreamingMovies']

# Normalise 'No internet service' → 'No'
df_s = df.copy()
for col in service_cols:
    df_s[col] = df_s[col].replace('No internet service', 'No')

churn_by_service = {}
for col in service_cols:
    rates = df_s.groupby(col)['Churn'].mean()
    churn_by_service[col] = rates.get('Yes', 0) - rates.get('No', 0)

pd.Series(churn_by_service).sort_values().plot(
    kind='barh', figsize=(7, 4), color='steelblue'
)
plt.title('Churn rate difference: with vs without each service')
plt.xlabel('Churn rate difference (positive = having service reduces churn)')
plt.axvline(0, color='gray', lw=0.8)
plt.tight_layout()
plt.show()

Security, backup, and tech support all reduce churn significantly — these are "stickiness" services. Streaming adds less, possibly because they're more discretionary.

In [ ]:
# Correlation heatmap — numeric only
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn']
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Numeric feature correlations')
plt.tight_layout()
plt.show()

TotalCharges and tenure are highly correlated (as expected — longer customers accumulate more charges). This means TotalCharges is partly redundant with tenure, but the charge-fulfillment ratio I'm engineering should capture the residual signal.

In [ ]:
# Multi-risk customer analysis: month-to-month + electronic check + fiber + short tenure
high_risk_mask = (
    (df['Contract'] == 'Month-to-month') &
    (df['PaymentMethod'] == 'Electronic check') &
    (df['tenure'] <= 12)
)
print(f"High-risk segment size: {high_risk_mask.sum()} customers")
print(f"High-risk churn rate: {df.loc[high_risk_mask, 'Churn'].mean():.2%}")
print(f"Overall churn rate: {df['Churn'].mean():.2%}")

The multi-risk segment confirms the hypothesis. This justifies creating a stacked risk flag as a feature — it captures a meaningful interaction that tree models might miss in early splits.

In [ ]:
# Tenure buckets — validating the bin choices
df['tenure_bucket_label'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 36, 60, float('inf')],
    labels=['0-12mo', '13-36mo', '37-60mo', '60+mo'],
    right=True
)
bucket_churn = df.groupby('tenure_bucket_label', observed=True)['Churn'].agg(['mean', 'count'])
bucket_churn.columns = ['churn_rate', 'count']
print(bucket_churn)

Bucket choice justified: churn drops significantly at each tier. 0-12 month customers are ~4x more likely to churn than 60+ month customers.

In [ ]:
# Senior citizen breakdown
print(df.groupby('SeniorCitizen')['Churn'].agg(['mean', 'count']))

Senior citizens churn at ~41% vs ~24% for non-seniors. Small group but meaningful signal.

In [ ]:
# Charge per service — engineering validation
service_cols_binary = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 
                       'OnlineBackup', 'DeviceProtection', 'TechSupport', 
                       'StreamingTV', 'StreamingMovies']

df_s2 = df.copy()
for col in service_cols_binary:
    df_s2[col] = df_s2[col].replace({'No internet service': 'No', 'No phone service': 'No'})
    df_s2[col] = (df_s2[col] == 'Yes').astype(int)

df_s2['num_services'] = df_s2[service_cols_binary].sum(axis=1)
df_s2['charge_per_service'] = df_s2['MonthlyCharges'] / (df_s2['num_services'] + 1)

fig, ax = plt.subplots(figsize=(8, 4))
df_s2.boxplot(column='charge_per_service', by='Churn', ax=ax)
ax.set_title('Charge per service by churn')
ax.set_xlabel('Churn (0=No, 1=Yes)')
ax.set_ylabel('$/service/month')
plt.suptitle('')
plt.tight_layout()
plt.show()

Good — charge per service does differ by churn status. Churners pay more per active service, suggesting they're not getting value from add-ons. This derived feature should add signal beyond raw MonthlyCharges.

Enough EDA. Key takeaways going into feature engineering:
- Contract type and tenure are the strongest signals
- Payment method (electronic check) is a risk indicator
- Fiber optic customers churn more — probably price sensitivity
- Security/backup/support add-ons reduce churn (stickiness)
- Multi-risk stacking works: month-to-month + short tenure + electronic check = very high churn
- Charge-per-service is worth engineering